# **Exploration**

## Questions
1. Import et choix de la source
2. Combien de pays distincts ? Combien de formats différents pour un même pays ?
3. Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?
4. Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?
5. Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?
6. Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

---
## 1. Import et choix de la source

### Imports des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

### Chargement des données

In [6]:
data_dir = Path("../data")

df_csv = pd.read_csv(data_dir / "numeros_tva.csv")
df_xlsx = pd.read_excel(data_dir / "numeros_tva.xlsx")

### Inspection et comparaison des données

In [ ]:
# Dimensions
print(f"Shape CSV: {df_csv.shape}")
print(f"Shape Excel: {df_xlsx.shape}")

# Types des colonnes
print("\nTypes CSV:\n", df_csv.dtypes)
print("\nTypes Excel:\n", df_xlsx.dtypes)

# Égalité stricte
print("\nSont-ils strictement identiques ?", df_csv.equals(df_xlsx))

Shape CSV: (10000, 6)
Shape Excel: (10000, 6)

Types CSV:
 id                int64
raison_sociale      str
pays_declare        str
numero_tva          str
date_saisie         str
source_saisie       str
dtype: object

Types Excel:
 id                int64
raison_sociale      str
pays_declare        str
numero_tva          str
date_saisie         str
source_saisie       str
dtype: object

Sont-ils strictement identiques ? True


### Choix de la source de donnée

In [10]:
# On attribue le .csv comme variable pour le DataFrame
df = df_csv

### **Réponses**

- Les deux fichiers contiennent 10 000 lignes et 6 colonnes avec le meme typage.
<br>
<br>
- **Choix de la source : `numero_tva.csv`**  
Un xlsx a plus de chance d'avoir été modifié. et le .csv est plus performant (lecture plus rapide et consommation mémoire plus légère).

---
## 2. Combien de pays distincts ? Combien de formats différents pour un même pays ?

### Afficher les différents pays distincts

In [24]:
df["pays_declare"].value_counts()

pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

### Calculer le nombre de pays distincts

In [25]:
df["pays_declare"].nunique()

15

### Formats différents de TVA pour la France "FR"

In [26]:
filtre_fr = df[df["pays_declare"] == "FR"]

In [27]:
longueurs_fr = df[df["pays_declare"] == "FR"]["numero_tva"].str.len().value_counts()
print(longueurs_fr)

numero_tva
13.0    624
14.0    142
16.0     56
11.0     47
15.0     46
12.0     32
1.0      10
17.0      1
Name: count, dtype: int64


In [28]:
df[df["pays_declare"] == "FR"]["numero_tva"].str.len().nunique()

8

In [29]:
# Les 2 premiers caractères des numéros pour le pays déclaré FR
prefixes_fr = df[df["pays_declare"].str.upper() == "FR"]["numero_tva"].str[:2].nunique()
print(prefixes_fr)

44


### **Réponses**

- **Nombre de `pays_declare` : 15**  
  - Mais certains ne sont pas de vrais pays (ZZ, QQ, XX)  
  - Et d'autres sont en dehors de l'Union Européenne (GB et UK)  
- **Nombre de pays distincts conservés : 10** (FR DK BE LU SE PT NL IT PL FI )
<br>
<br>
<br>
- Pour la France `pays_declare = "FR"` on observe
    - **8 longueurs de chaînes distinctes**  
    Numéros de 1 à 17 caractères, alors que le standard est de 13 caractères.
    - **44 préfixes différents**  
    Numéros démarrent avec 44 préfixes différents au lieu du seul FR attendu : présence de minuscules fr, d'espaces au début, de tirets, ou de chiffres directs sans code pays.

---
## 3. Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

### Calculer les nombre de lignes avec probleme de caratére dans `numero_tva`

In [33]:
ligne_polluee = df["numero_tva"].str.contains(r"[^a-zA-Z0-9]", regex=True)

nb_lignes_ligne_polluees = ligne_polluee.sum()
proportion = ligne_polluee.mean() * 100

print(f"Nombre de lignes polluées : {nb_lignes_ligne_polluees}")
print(f"Proportion : {proportion:.2f} %")

Nombre de lignes polluées : 1813
Proportion : 18.13 %


### Afficher les caractères parasites

In [34]:
caracteres_parasites = df["numero_tva"].dropna().str.findall(r"[^a-zA-Z0-9]").explode().value_counts()
print(caracteres_parasites)

numero_tva
     2236
-     471
.     404
Name: count, dtype: int64


### **Réponses**

- **Nombre de lignes polluées : 1813**
<br>
<br>
- **Proportion : 18.13 %**
<br>
<br>
- Les caractéres parasites :
  - L'espace " " : 2236 occurrences
  - Le tiret "-" : 471 occurrences
  - Le point "." : 404 occurrences

---
## 4. Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

### Calculer et afficher les valeurs vides différentes dans `numéro_tva`

In [49]:
# On charge les données brutes sans transformation de pandas
df_brut = pd.read_csv("../data/numeros_tva.csv", keep_default_na=False)

# On garde tout ce qui ne contient pas de chiffre
sans_chiffres = df_brut[~df_brut["numero_tva"].str.contains(r"\d", regex=True)]["numero_tva"]

# On renomme le vide et les espaces pour un affichage lisible
formes_lisibles = sans_chiffres.apply(lambda x: "[Chaîne vide]" if x == "" else ("[Espaces seuls]" if x.isspace() else x))

# On affiche chaque forme ligne par ligne
decompte = formes_lisibles.value_counts()
for forme, nb in decompte.items():
    print(f"Valeur {forme} : {nb}")

print("\nTotal des valeurs vides : ", len(sans_chiffres))

Valeur [Chaîne vide] : 60
Valeur [Espaces seuls] : 59
Valeur - : 55
Valeur N/A : 48
Valeur null : 38
Valeur NU.LL : 1

Total des valeurs vides :  261


### **Réponses**

- **Total des valeurs vides :  261**
    - Valeur [Chaîne vide] : 60 occurrences
    - Valeur [Espaces seuls] : 59 occurrences
    - Valeur - : 55 occurrences
    - Valeur N/A : 48 occurrences
    - Valeur null : 38 occurrences
    - Valeur NU.LL : 1 occurrence

---
## 5. Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?

### Calculer et afficher le nombre de `pays_declare` avec un préfixe différents dans `numero_tva`

In [53]:
# Extraction des 2 premiers caractères de numéro_tva
prefixe_tva = df["numero_tva"].str.strip().str.upper().str[:2]

# Prefixe tva différent du pays déclaré 
incoherences = df[df["pays_declare"] != prefixe_tva]

# Nombre de ligne et proportion
print(f"Nombre de lignes où le préfixe ne correspond pas au pays déclaré : {len(incoherences)}")
print(f"Proportion : {len(incoherences) / len(df) * 100:.2f} %")

Nombre de lignes où le préfixe ne correspond pas au pays déclaré : 793
Proportion : 7.93 %


### **Réponses**

- **Non, le pays déclaré ne correspond pas toujours au préfixe du numéro de TVA.**
    - Nombre d'incohérences : 793
    - Proportion d'incohérences : 7.93 %

---
## 6. Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

### Calculer et afficher le nombre de `source_saisie` différentes

In [54]:
df["source_saisie"].value_counts()

source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

### Calculer et afficher les `date_saisie` différentes

In [65]:
# Convertir en datetime et analyser par année
df['date_saisie'] = pd.to_datetime(df['date_saisie'])

# Répartition par année
print("--- Répartition par année ---")
print(df['date_saisie'].dt.year.value_counts().sort_index())

# Min et Max des dates
print(f"\nPériode couverte : du {df['date_saisie'].min().strftime('%d/%m/%Y')} au {df['date_saisie'].max().strftime('%d/%m/%Y')}")

--- Répartition par année ---
date_saisie
2024    5760
2025    4240
Name: count, dtype: int64

Période couverte : du 01/01/2024 au 01/10/2025


### Calculer et afficher le taux d'erreur de saisie par canal de saisie

In [64]:
df['est_vide'] = ~df['numero_tva'].astype(str).str.contains(r'\d')
taux_vides_canal = df.groupby('source_saisie')['est_vide'].mean() * 100
print("--- Taux de numéros vides par canal (%) ---")
print(taux_vides_canal.round(2))

--- Taux de numéros vides par canal (%) ---
source_saisie
crm                   2.82
import_fournisseur    2.88
portail_client        2.36
reprise_erp           2.85
saisie_manuelle       2.13
Name: est_vide, dtype: float64


In [67]:
# 1. Définition des indicateurs d'anomalies
df["pays_inconnus"] = df["pays_declare"].isin(["ZZ", "QQ", "XX"])
df["pays_hors_ue"] = df["pays_declare"].isin(["GB", "UK"])
df["tva_vide"] = ~df["numero_tva"].astype(str).str.contains(r"\d")
df["tva_non_conforme"] = df["numero_tva"].astype(str).str.contains(r"[^a-zA-Z0-9]")

# 2. Regroupement et calcul des pourcentages par canal de saisie
bilan_par_canal = df.groupby("source_saisie")[
    ["pays_inconnus", "pays_hors_ue", "tva_vide", "tva_non_conforme"]
].mean() * 100

# 3. Renommage des colonnes pour un affichage propre
bilan_par_canal.columns = [
    "Pays inconnus (%)",
    "Hors-UE (GB/UK) (%)",
    "Numéros de TVA vides (%)",
    "Numéro de TVA non conforme (%)"
]

print("--- Taux d'anomalies par canal de saisie (%) ---")
display(bilan_par_canal.round(2))

--- Taux d'anomalies par canal de saisie (%) ---


,Pays inconnus (%),Hors-UE (GB/UK) (%),Numéros de TVA vides (%),Numéro de TVA non conforme (%)
source_saisie,,,,
crm,3.22,2.72,2.82,18.62
import_fournisseur,3.29,1.34,2.88,19.08
portail_client,2.86,1.81,2.36,17.59
reprise_erp,3.24,2.27,2.85,18.30
saisie_manuelle,2.94,2.23,2.13,17.05


In [79]:
# Répartition par mois
df['date_saisie'] = pd.to_datetime(df['date_saisie'])
repartition_mensuelle = df['date_saisie'].dt.to_period('M').value_counts().sort_index()

print("--- Répartition par mois ---")
print(repartition_mensuelle)

# Mois le plus fort et le plus faible
mois_max = repartition_mensuelle.idxmax()
val_max = repartition_mensuelle.max()

mois_min = repartition_mensuelle.idxmin()
val_min = repartition_mensuelle.min()

print("\n--- Statistiques mensuelles ---")
print(f"Mois le plus fort : {mois_max} avec {val_max} saisies")
print(f"Mois le plus faible : {mois_min} avec {val_min} saisies")
print(f"Moyenne mensuelle : {repartition_mensuelle.mean():.1f} saisies")

--- Répartition par mois ---
date_saisie
2024-01    461
2024-02    446
2024-03    504
2024-04    510
2024-05    489
2024-06    503
2024-07    469
2024-08    489
2024-09    443
2024-10    458
2024-11    494
2024-12    494
2025-01    497
2025-02    428
2025-03    456
2025-04    465
2025-05    457
2025-06    468
2025-07    500
2025-08    508
2025-09    445
2025-10     16
Freq: M, Name: count, dtype: int64

--- Statistiques mensuelles ---
Mois le plus fort : 2024-04 avec 510 saisies
Mois le plus faible : 2025-10 avec 16 saisies
Moyenne mensuelle : 454.5 saisies


### **Réponses**

- **Canal de saisie `source_saisie` : Aucun ne sort du lot**
    - Les 5 sources ont un taux de saisis équivalent (~20 % chacun, soit ~2 000 enregistrements).
    - Les 5 sources ont un taux d'anomalies équivalent (~3 % de pays inconnus, ~2 % hors-UE, ~2,5 % de numéro de TVA vides et ~18 % de numéro de TVA non conformes).
<br>
<br>
- **Périodes de saisie `date_saisie` : L'année 2025 et son mois d'octobre sortent du lot car ils sont incompléts**
    - L'historique s'étend sur 21 mois, du 01/01/2024 au 01/10/2025. 12 mois pour 2024 et 10 mois pour 2025.
    - La moyenne mensuelle de saisie est de ~475 saisies par mois complet (pic à 510 en avril 2024 et creux à 428 en février 2025).
    - Octobre 2025 sort du lot avec seulement 16 saisies (1 seul jour de saisie).